In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine(
    os.getenv("DATASET_DB_URL", "mysql+pymysql://root:@127.0.0.1:3307/family_finance_mono?charset=utf8mb4")
)

In [2]:
# Gastos e ingresos
df = pd.read_sql("""
SELECT
   combined.year_month,
   BIN_TO_UUID(combined.family_id_bin) AS family_id,
   BIN_TO_UUID(combined.family_member_id_bin) AS family_member_id,
   SUM(combined.total_income) AS total_income,
   SUM(combined.total_expenses) AS total_expenses
FROM (
   SELECT
      DATE_FORMAT(ie.date, '%%Y-%%m') AS `year_month`,
      fm.family_id AS family_id_bin,
      ie.family_member_id AS family_member_id_bin,
      SUM(ie.amount) AS total_income,
      0 AS total_expenses
   FROM income_entries ie
   LEFT JOIN family_members fm ON ie.family_member_id = fm.id
   WHERE ie.is_active = 1
   GROUP BY
      DATE_FORMAT(ie.date, '%%Y-%%m'),
      fm.family_id,
      ie.family_member_id

   UNION ALL

   SELECT
      DATE_FORMAT(e.date, '%%Y-%%m') AS `year_month`,
      fm.family_id AS family_id_bin,
      e.family_member_id AS family_member_id_bin,
      0 AS total_income,
      SUM(e.amount) AS total_expenses
   FROM expenses e
   LEFT JOIN family_members fm ON e.family_member_id = fm.id
   GROUP BY
      DATE_FORMAT(e.date, '%%Y-%%m'),
      fm.family_id,
      e.family_member_id
) AS combined
GROUP BY
   combined.year_month,
   combined.family_id_bin,
   combined.family_member_id_bin
ORDER BY
   combined.year_month,
   combined.family_id_bin,
   combined.family_member_id_bin
""", engine)



In [3]:
# Sacamos el balance
df['balance'] = df['total_income'] - df['total_expenses']

In [4]:
df.head(10)

,year_month,family_id,family_member_id,total_income,total_expenses,balance
0,2022-01,b006e9da-0024-426a-8f7d-dfa84d676ff7,48777ba1-af72-474e-8a7b-3f8aa1027136,0.00,346.81,-346.81
1,2022-01,d0000000-0000-0000-0000-000000000000,32c23e4d-1673-11f1-854c-669f06e2d810,0.00,296.31,-296.31
2,2022-01,d0000000-0000-0000-0000-000000000000,32c24029-1673-11f1-854c-669f06e2d810,0.00,616.35,-616.35
3,2022-01,d0000000-0000-0000-0000-000000000000,32c2404c-1673-11f1-854c-669f06e2d810,0.00,969.22,-969.22
4,2022-01,d0000000-0000-0000-0000-000000000000,ea000000-0000-0000-0000-000000000000,1261.72,585.50,676.22
5,2022-01,d1111111-1111-1111-1111-111111111111,32c24081-1673-11f1-854c-669f06e2d810,9087.55,1682.80,7404.75
6,2022-01,d1111111-1111-1111-1111-111111111111,32c24098-1673-11f1-854c-669f06e2d810,5919.37,1927.90,3991.47
7,2022-01,d1111111-1111-1111-1111-111111111111,32c240af-1673-11f1-854c-669f06e2d810,0.00,981.47,-981.47
8,2022-01,d1111111-1111-1111-1111-111111111111,e1111111-1111-1111-1111-111111111111,0.00,2521.39,-2521.39
9,2022-01,d2222222-2222-2222-2222-222222222222,32c240c8-1673-11f1-854c-669f06e2d810,0.00,256.33,-256.33


In [ ]:
# Extraemos los datos a un csv
df.to_csv('../../data/family-finance-member-data.csv', index=False)